Use this code to check if the ground truth water depth is the same as mSWEGNN ground truth water depth

NOTE: Do not forget to change the target variable to water_depth in FloodEventDataset

In [39]:
import os
import torch
import numpy as np
import pandas as pd

from data import dataset_factory
from data.dem_data_retrieval import get_filled_dem, get_aspect, get_curvature, get_flow_accumulation
from data.shp_data_retrieval import get_cell_position
from torch_geometric.loader import DataLoader
from utils import file_utils

In [40]:
config_path = 'configs/mswegnn_config.yaml'
config = file_utils.read_yaml_file(config_path)

In [41]:
# Create additional DEMs beforehand
dataset_parameters = config['dataset_parameters']
root_dir = dataset_parameters['root_dir']
dataset_summary_file = dataset_parameters['testing']['dataset_summary_file']

# dataset_summary_path = os.path.join(root_dir, 'raw', dataset_summary_file)
# dataset_summary_df = pd.read_csv(dataset_summary_path)

# for idx, row in dataset_summary_df.iterrows():
#     run_id = row['Run_ID']
#     dem_file = row['DEM_Filepath']
#     nodes_shp_file = row['Nodes_Shp_Filepath']

#     dem_path = os.path.join(root_dir, 'raw', dem_file)
#     nodes_shp_path = os.path.join(root_dir, 'raw', nodes_shp_file)
#     dem_filename = os.path.splitext(os.path.basename(dem_path))[0]
#     dem_folder = os.path.dirname(dem_path)

#     pos = get_cell_position(nodes_shp_path)

#     print(f'Processing DEM for Run ID {run_id} ({dem_path})')
#     filled_dem_path = os.path.join(dem_folder, f'{dem_filename}_filled.tif')
#     filled_dem = get_filled_dem(dem_path, filled_dem_path)

#     aspect_dem_path = os.path.join(dem_folder, f'{dem_filename}_aspect.tif')
#     if not os.path.exists(aspect_dem_path):
#         get_aspect(filled_dem, aspect_dem_path, pos)

#     curvature_dem_path = os.path.join(dem_folder, f'{dem_filename}_curvature.tif')
#     if not os.path.exists(curvature_dem_path):
#         get_curvature(filled_dem, curvature_dem_path, pos)

#     flow_dir_dem_path = os.path.join(dem_folder, f'{dem_filename}_flow_dir.tif')
#     flow_acc_dem_path = os.path.join(dem_folder, f'{dem_filename}_flow_acc_dem.tif')
#     if not os.path.exists(flow_acc_dem_path):
#         get_flow_accumulation(filled_dem, flow_dir_dem_path, flow_acc_dem_path, pos)

In [42]:
# Create test dataset
event_stats_file = dataset_parameters['testing']['event_stats_file']
previous_timesteps = dataset_parameters['previous_timesteps']

dataset_config = {
    'mode': 'test',
    'root_dir': root_dir,
    'dataset_summary_file': dataset_summary_file,
    'event_stats_file': event_stats_file,
    'features_stats_file': dataset_parameters['features_stats_file'],
    'previous_timesteps': previous_timesteps,
    'normalize': False, # No normalization for the ground truth comparison
    'timestep_interval': dataset_parameters['timestep_interval'],
    'spin_up_time': dataset_parameters['spin_up_time'],
    'time_from_peak': dataset_parameters['time_from_peak'],
    'inflow_boundary_nodes': dataset_parameters['inflow_boundary_nodes'],
    'outflow_boundary_nodes': dataset_parameters['outflow_boundary_nodes'],
    'with_global_mass_loss': False,
    'with_local_mass_loss': False,
    'force_reload': True,
}
dataset = dataset_factory(storage_mode='memory', autoregressive=False, **dataset_config)

Processing Flood Event Dataset...
Converting .xyz to .tif file at: c:\Users\Carlo\Documents\School\Masters\NUS\Dissertation\flood_pi_gnn\data_mswegnn\datasets\raw\DEM\DEM_96.tif
Creating filled DEM at: c:\Users\Carlo\Documents\School\Masters\NUS\Dissertation\flood_pi_gnn\data_mswegnn\datasets\raw\DEM\DEM_96_filled.tif
Creating aspect DEM at: c:\Users\Carlo\Documents\School\Masters\NUS\Dissertation\flood_pi_gnn\data_mswegnn\datasets\raw\DEM\DEM_96_aspect.tif
Creating curvature DEM at: c:\Users\Carlo\Documents\School\Masters\NUS\Dissertation\flood_pi_gnn\data_mswegnn\datasets\raw\DEM\DEM_96_curvature.tif
Creating flow direction DEM at: c:\Users\Carlo\Documents\School\Masters\NUS\Dissertation\flood_pi_gnn\data_mswegnn\datasets\raw\DEM\DEM_96_flow_dir.tif
Creating flow accumulation DEM at: c:\Users\Carlo\Documents\School\Masters\NUS\Dissertation\flood_pi_gnn\data_mswegnn\datasets\raw\DEM\DEM_96_flow_acc_dem.tif
Converting .xyz to .tif file at: c:\Users\Carlo\Documents\School\Masters\NUS\Di

Processing timesteps: 100%|██████████| 235/235 [00:00<00:00, 1541.37it/s]


RAM usage after loading dataset: 1.43 GB


In [43]:
def get_event_ground_truth(event_dataset):
    sliding_window_length = previous_timesteps + 1
    target_nodes_idx = event_dataset.DYNAMIC_NODE_FEATURES.index(event_dataset.NODE_TARGET_FEATURE)
    start_node_target_idx = event_dataset.num_static_node_features + (target_nodes_idx * sliding_window_length)
    end_node_target_idx = start_node_target_idx + sliding_window_length

    ground_truths = []
    dataloader = DataLoader(event_dataset, batch_size=1, shuffle=False) # Enforce batch size = 1 for autoregressive testing
    for graph in dataloader:
        label = graph.x[:, [end_node_target_idx-1]] + graph.y
        label = torch.clip(label, min=0)
        non_boundary_nodes_mask = ~graph.boundary_nodes_mask
        label = label[non_boundary_nodes_mask]
        ground_truths.append(label)
    ground_truths = torch.stack(ground_truths).numpy()
    return ground_truths

def get_saved_wd_filename(run_id: str) -> str:
    return f'water_depth_gt_run_{run_id}.npy'

def get_mswegnn_val_filename(run_id: str) -> str:
    return f'mSWEGNN_runid_{run_id}_metrics.npz'

In [44]:
SAVE_DIR_PATH = 'saved_metrics/water_depth_gt'
os.makedirs(SAVE_DIR_PATH, exist_ok=True)

for event_idx, run_id in enumerate(dataset.event_run_ids):
    print(f'Getting ground truth for run {event_idx + 1}/{len(dataset.event_run_ids)} with Run ID {run_id}')

    event_start_idx = dataset.event_start_idx[event_idx]
    event_end_idx = dataset.event_start_idx[event_idx + 1] if event_idx + 1 < len(dataset.event_start_idx) else dataset.total_rollout_timesteps
    event_dataset = dataset[event_start_idx:event_end_idx]
    ground_truth = get_event_ground_truth(event_dataset)

    filename = get_saved_wd_filename(run_id)
    save_path = os.path.join(SAVE_DIR_PATH, filename)
    print(f'Saving ground truth water depths to {save_path}')
    np.save(save_path, ground_truth)

Getting ground truth for run 1/5 with Run ID 96
Saving ground truth water depths to saved_metrics/water_depth_gt\water_depth_gt_run_96.npy
Getting ground truth for run 2/5 with Run ID 97
Saving ground truth water depths to saved_metrics/water_depth_gt\water_depth_gt_run_97.npy
Getting ground truth for run 3/5 with Run ID 98
Saving ground truth water depths to saved_metrics/water_depth_gt\water_depth_gt_run_98.npy
Getting ground truth for run 4/5 with Run ID 99
Saving ground truth water depths to saved_metrics/water_depth_gt\water_depth_gt_run_99.npy
Getting ground truth for run 5/5 with Run ID 100
Saving ground truth water depths to saved_metrics/water_depth_gt\water_depth_gt_run_100.npy


In [45]:
EPS = 1e-1
MSWEGNN_VAL_DIR_PATH = 'mswegnn_baselines'
for file in os.listdir(SAVE_DIR_PATH):
    run_id = file.split('_')[-1].split('.')[0]

    event_wd_gt = np.load(os.path.join(SAVE_DIR_PATH, file))

    mswegnn_filename = get_mswegnn_val_filename(run_id)
    mswegnn_path = os.path.join(MSWEGNN_VAL_DIR_PATH, mswegnn_filename)
    mswegnn_val_gt = np.load(mswegnn_path)['target']

    # Remove previous timesteps from mswegnn_val_gt
    mswegnn_val_gt = mswegnn_val_gt[previous_timesteps:]

    print(f'Checking target for Run ID {run_id}')
    assert np.all((event_wd_gt - mswegnn_val_gt) < EPS), f'Targets are not equal for Run ID {run_id}'

Checking target for Run ID 100
Checking target for Run ID 81
Checking target for Run ID 82
Checking target for Run ID 83
Checking target for Run ID 84
Checking target for Run ID 85
Checking target for Run ID 86
Checking target for Run ID 87
Checking target for Run ID 88
Checking target for Run ID 89
Checking target for Run ID 90
Checking target for Run ID 91
Checking target for Run ID 92
Checking target for Run ID 93
Checking target for Run ID 94
Checking target for Run ID 95
Checking target for Run ID 96
Checking target for Run ID 97
Checking target for Run ID 98
Checking target for Run ID 99
